# Computational modelling of hiPSC-derived neuronal networks
## Part 1: Exploring the network model
**Neuroengineering Summer School 'Massimo Grattarola', Camogli 2026**

---

In this notebook we build a biophysical model of hiPSC-derived
neuronal network on a MEA, run simulations, and compare the output to
"experimental recordings" from a healthy- and a patient-derived culture.

The patient-derived culture is from a patient with a mutation in KCNQ2.
KCNQ2 is a gene encoding a voltage-gated potassium channel (the M-channel)
and is associated with a neonatal epileptic encephalopathy.
(note: this is not real experimental data, but mimics all aspects of it)

**Structure:**
1. Setup notebook and dependencies
2. Summary features
3. The network model (play around)
4. The experimental recordings (assess)
5. Exercise: find the KCNQ2 parameters

9## 1. Setup

In [ ]:
!pip install brian2 -q
!wget -q https://raw.githubusercontent.com/ninontwik/NeuroEngSummerSchool2026/main/data/Healtycontrol_MEArecording.csv
!wget -q https://raw.githubusercontent.com/ninontwik/NeuroEngSummerSchool2026/main/data/KCNQ2patient_MEArecording.csv
print('Setup complete.')

In [ ]:
import builtins
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *

BrianLogger.suppress_hierarchy('brian2')
seed(42)
np.random.seed(42)

## 2. Summary features

To compare simulations to experimental recordings quantitatively, we reduce
each spike train dataset to a small set of scalar **summary features**.
These same features will be used as summary statistics for SBI in notebook 2.

We provide six features below. As you work through the notebook, think about
what each one captures -- and whether all of them are equally useful for
distinguishing healthy from disease dynamics, and for fitting the model.

In [ ]:
def compute_features(spike_data, n_units, sim_duration, bin_size=0.05):
    """
    Compute summary features from a two-column spike data array.

    Parameters
    ----------
    spike_data : np.ndarray, shape (n_spikes, 2)
        Column 0: unit index, column 1: spike time (seconds).
    n_units : int
        Total number of units (neurons/electrodes) including silent ones.
    sim_duration : float
        Recording duration in seconds.
    bin_size : float
        Bin width in seconds for population firing rate (default 50 ms).

    Returns
    -------
    dict mapping feature name to float value.
    """
    bin_edges = np.arange(0, sim_duration + bin_size, bin_size)
    n_bins = builtins.len(bin_edges) - 1

    # -- Per-unit firing rates (Hz) ------------------------------------------
    # Count spikes for each unit and divide by recording duration.
    unit_rates = np.zeros(n_units)
    for i in builtins.range(n_units):
        unit_rates[i] = int(np.sum(spike_data[:, 0] == i)) / sim_duration

    # FEATURE 1: Mean firing rate (Hz)
    # Average firing rate across all units. Captures overall excitability.
    feat_mean_fr = np.mean(unit_rates)

    # FEATURE 2: Number of active units (
    # Number of units that fired at least one spike.
    feat_n_active = int(np.sum(unit_rates > 0))

    # FEATURE 3: CV of firing rate across units
    # Coefficient of variation (std/mean) of per-unit firing rates.
    # High CV = heterogeneous activity across the network.
    feat_cv_fr = np.std(unit_rates) / (np.mean(unit_rates) + 1e-9)

    # -- Bin spikes into a rate matrix [n_units x n_bins] --------------------
    # Each row is the firing rate trace of one unit over time.
    # We need this for the population rate and correlation features.
    rate_matrix = np.zeros((n_units, n_bins))
    for i in builtins.range(n_units):
        spike_times_i = spike_data[spike_data[:, 0] == i, 1]
        if builtins.len(spike_times_i) > 0:
            counts, _ = np.histogram(spike_times_i, bins=bin_edges)
            rate_matrix[i] = counts / bin_size  # convert counts to Hz

    # Population firing rate: total spikes across all units per time bin
    pop_rate = np.sum(rate_matrix, axis=0)  # shape: (n_bins,)

    # FEATURE 4: Variance of population firing rate over time (Hz^2)
    # High variance = strong fluctuations = bursty activity.
    # Low variance = tonic, regular firing.
    # This captures burstiness (a little) without explicit burst detection.
    feat_poprate_var = np.var(pop_rate)

    # FEATURE 5: Early vs late firing rate ratio
    # Ratio of mean population rate in the first vs second half of the recording.
    half = n_bins // 2
    feat_early_late = np.mean(pop_rate[:half]) / (np.mean(pop_rate[half:]) + 1e-9)

    # FEATURE 6: Mean pairwise correlation coefficient
    # Pearson correlation between all pairs of unit firing rate traces.
    # High correlation = synchronised network activity.
    # Only computed for units with enough spikes to be meaningful.
    active_mask = unit_rates > 0.05  # at least ~3 spikes in 60 s
    active_rates = rate_matrix[active_mask]
    if active_rates.shape[0] > 1:
        corr_matrix = np.corrcoef(active_rates)
        upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
        feat_mean_corr = float(np.mean(upper_tri))
    else:
        feat_mean_corr = 0.0

    return {
        'mean_firing_rate'   : float(feat_mean_fr),
        'n_active_units'     : float(feat_n_active),
        'cv_firing_rate'     : float(feat_cv_fr),
        'poprate_variance'   : float(feat_poprate_var),
        'early_late_ratio'   : float(feat_early_late),
        'mean_pairwise_corr' : feat_mean_corr,
    }

## 3. The network model

We model the culture as a network of **Adaptive Exponential Integrate-and-Fire
(AdEx)** neurons. The AdEx model extends the standard leaky integrate-and-fire
model with two additions:
- An **exponential term** that produces a realistic action potential upswing
- A **spike-frequency adaptation current** $w$ that mimics the M-current

The membrane potential of each neuron evolves as:

$$C_m \frac{dv}{dt} = g_L(E_{rest} - v) + g_L \Delta_T \exp\!\left(\frac{v-V_T}{\Delta_T}\right) - w + I_{syn}$$

The adaptation current decays between spikes and is incremented by $b$ after
each spike, gradually reducing excitability during a burst until it terminates.

**Three parameters are left free (unknown)** -- these are what we want to infer:

| Parameter | Symbol | Biological meaning                                                             |
|---|---|--------------------------------------------------------------------------------|
| `B_ADAPT` | $b$ (pA) | Adaptation increment per spike -- strength of M-current                        |
| `POISSON_RATE` | $r$ (Hz) | Background input rate -- ambient excitability and spontaneous synaptic release |
| `G_SYN` | $g$ (nS) | Recurrent synaptic weight -- E to E connection strength                        |

For all other parameters, assume we measured the values experimentally (values provided).

In [ ]:
# =============================================================================
# SECTION 1: MODEL PARAMETERS
# Parameters marked FIXED are set to "experimentally measured" values.
# Parameters marked FREE are the ones we want to infer from data.
# Feel free to explore the effect of the fixed parameters too -- this gives
# you a sense of how rich the full parameter space is even for this simple model.
# =============================================================================

# -- Simulation settings ------------------------------------------------------
SIMULATION_TIME = 60 * second   # total simulation duration
TRANSIENT_TIME  = 2  * second   # warm-up before recording: lets the network
                                # settle into its characteristic state
DT = 0.1 * ms                   # integration timestep

# -- Network structure (FIXED) ------------------------------------------------
N_NEURONS       = 50            # number of excitatory neurons
CONNECTION_PROB = 0.15          # probability of a synapse between any two neurons

# -- AdEx neuron parameters (FIXED, measured experimentally) ------------------
#
#   C_m * dv/dt = g_L*(E_rest - v) + g_L*dT*exp((v-V_T)/dT) - w + I_syn
#
C_m     = 100  * pF    # membrane capacitance
g_L     = 10   * nS    # leak conductance
E_L     = -50  * mV    # resting potential (mean; each neuron gets a slightly different value)
V_T     = -40  * mV    # spike initiation threshold
DELTA_T = 2    * mV    # sharpness of action potential upswing
V_SPIKE = 20   * mV    # voltage at which a spike is detected and reset triggered
V_RESET = -65  * mV    # post-spike reset potential

# -- Spike-frequency adaptation  ---------------------------------------------
# After each spike, the adaptation current w is incremented by B_ADAPT.
# w then decays back to zero with time constant TAU_W.
# This creates a braking force that grows during a burst and eventually
# terminates it
TAU_W  = 6000 * ms     # (FIXED) adaptation (sAHP) time constant
B_ADAPT = 50  * pA     # (FREE)  adaptation increment per spike

# -- Background Poisson input -------------------------------------------------
# Each neuron receives 100 independent Poisson spike trains, modelling
# spontaneous synaptic release and tonic drive from non-spiking cells.
POISSON_RATE   = 1000 * Hz   # (FREE)  rate of background input
POISSON_WEIGHT = 0.01 * nS   # (FIXED) synaptic weight of each background event

# -- Recurrent excitatory synapses (FIXED) ------------------------------------
# AMPA-like single-exponential conductance synapses.
# Synaptic current: I_syn = g_syn * (E_exc - v)
E_EXC   = 0   * mV    # excitatory reversal potential
TAU_SYN = 6   * ms    # synaptic decay time constant
G_SYN   = 2.0 * nS    # (FREE) peak synaptic conductance

# -- Neuron heterogeneity (FIXED) ---------------------------------------------
# Each neuron gets a slightly different resting potential drawn from a
# Gaussian, mimicking biological variability in the culture.
HETEROGENEITY_STD = 5 * mV

### Model equations

Brian2 uses string-based equations. Each line defines either a differential
equation (`d.../dt = ...`) or an auxiliary variable. The equations are
compiled and integrated numerically at each timestep.

In [ ]:
# =============================================================================
# SECTION 2: NEURON MODEL EQUATIONS
# Brian2 equation strings do not support inline comments, so the explanation
# is given below.
# =============================================================================

neuron_equations = """
    dv/dt = (g_L*(E_rest - v) + g_L*delta_T*exp((v - V_T)/delta_T) - w + I_syn) / C_m : volt
    dw/dt = -w / tau_w : amp
    dg_syn/dt = -g_syn / tau_syn : siemens
    I_syn = g_syn * (E_exc - v) : amp
    E_rest : volt
"""
# dv/dt   : AdEx membrane equation. The exp() term produces the sharp spike
#           upswing. E_rest is per-neuron (see initial conditions below).
# dw/dt   : Adaptation current. Decays exponentially between spikes,
#           incremented by B_ADAPT after each spike (see reset string).
# dg_syn  : Synaptic conductance. Incremented by incoming spikes (recurrent
#           and Poisson), decays with TAU_SYN.
# I_syn   : Synaptic current. Drives v towards E_EXC when g_syn > 0.
# E_rest  : Per-neuron resting potential. Set in initial conditions below.

### Building and running the network

In [ ]:
# =============================================================================
# SECTION 3: BUILD THE NETWORK
# =============================================================================

start_scope()           # reset Brian2 state (important when re-running cells)
seed(42)
np.random.seed(42)
defaultclock.dt = DT

# -- Create neuron population -------------------------------------------------
# threshold : spike is triggered when v crosses V_SPIKE
# reset     : after a spike, v is reset and adaptation current is incremented
neurons = NeuronGroup(
    N_NEURONS,
    model=neuron_equations,
    threshold='v > V_SPIKE',
    reset='v = V_RESET; w += b_adapt',
    method='euler',
    namespace={
        'g_L': g_L, 'delta_T': DELTA_T, 'V_T': V_T, 'C_m': C_m,
        'tau_w': TAU_W, 'b_adapt': B_ADAPT,
        'E_exc': E_EXC, 'tau_syn': TAU_SYN,
    }
)

# -- Initial conditions -------------------------------------------------------
# Each neuron gets its own resting potential drawn from N(E_L, HETEROGENEITY_STD).
# v is initialised to E_rest so every neuron starts at rest.
neurons.E_rest = E_L + HETEROGENEITY_STD * np.random.randn(N_NEURONS)
neurons.v      = neurons.E_rest
neurons.w      = 0 * pA
neurons.g_syn  = 0 * nS

# -- Background Poisson input -------------------------------------------------
# Each neuron receives N=100 independent Poisson sources, each incrementing
# g_syn by POISSON_WEIGHT on each event.
poisson_input = PoissonInput(
    target=neurons, target_var='g_syn',
    N=100, rate=POISSON_RATE, weight=POISSON_WEIGHT
)

# -- Recurrent excitatory synapses --------------------------------------------
# Random connectivity with probability CONNECTION_PROB.
# Each presynaptic spike increments the postsynaptic g_syn by G_SYN.
synapses = Synapses(
    source=neurons, target=neurons,
    on_pre='g_syn_post += G_SYN',
    namespace={'G_SYN': G_SYN}
)
synapses.connect(p=CONNECTION_PROB)

print(f'Network created: {N_NEURONS} neurons, '
      f'{builtins.len(synapses.i[:])} synapses')

In [ ]:
# =============================================================================
# SECTION 4: RUN SIMULATION
# The simulation runs in two phases:
#   1. Warm-up (TRANSIENT_TIME): not recorded. Lets the network settle from
#      the artificial initial conditions into its equilibrium dynamic.
#   2. Recording (SIMULATION_TIME): the SpikeMonitor is active and
#      records the time and index of every spike.
# =============================================================================

spike_mon = SpikeMonitor(neurons)

# for optional exercise --------
# N_EXAMPLE_NEURONS = 5
# state_mon = StateMonitor(
#     neurons,
#     variables=["v", "w"],
#     record=builtins.list(builtins.range(N_EXAMPLE_NEURONS))
# )
# net.add(state_mon)  # uncomment if using the optional StateMonitor above

net = Network(neurons, poisson_input, synapses)

print('Running warm-up...')
net.run(TRANSIENT_TIME)

net.add(spike_mon)      # attach monitor only after warm-up
print('Running simulation...')
net.run(SIMULATION_TIME, report='text')

In [ ]:
# =============================================================================
# SECTION 5: EXTRACT SPIKE DATA
# Convert Brian2 SpikeMonitor output to a standard two-column array:
#   column 0 -- unit index  (int)
#   column 1 -- spike time in seconds  (float)
# Spike times are shifted so t=0 corresponds to the start of the
# recorded period (after the warm-up).
# =============================================================================

spike_data_sim = np.column_stack((
    np.array(spike_mon.i[:]),
    np.array(spike_mon.t / second) - TRANSIENT_TIME / second
))
spike_data_sim = spike_data_sim[spike_data_sim[:, 1].argsort()]

print(f'Total spikes recorded: {spike_data_sim.shape[0]}')
print(f'Array shape: {spike_data_sim.shape}  (n_spikes x 2)')

In [ ]:
# =============================================================================
# SECTION 6: PLOT RASTER
# =============================================================================

bin_size  = 0.05  # seconds
sim_dur   = SIMULATION_TIME / second
n_units   = N_NEURONS

bin_edges   = np.arange(0, sim_dur + bin_size, bin_size)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Compute population firing rate for the plot
rate_matrix = np.zeros((n_units, builtins.len(bin_edges) - 1))
for i in builtins.range(n_units):
    st = spike_data_sim[spike_data_sim[:, 0] == i, 1]
    if builtins.len(st) > 0:
        counts, _ = np.histogram(st, bins=bin_edges)
        rate_matrix[i] = counts / bin_size
pop_rate = np.sum(rate_matrix, axis=0)

fig, (ax_raster, ax_pop) = plt.subplots(
    2, 1, figsize=(10, 4), sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
)
ax_raster.plot(spike_data_sim[:, 1], spike_data_sim[:, 0],
               '|k', markersize=2, alpha=0.5)
ax_raster.set_ylabel('Unit index')
ax_raster.set_title(
    f'Simulation  |  b_adapt={B_ADAPT/pA:.0f} pA,  '
    f'poisson_rate={POISSON_RATE/Hz:.0f} Hz,  '
    f'g_syn={G_SYN/nS:.1f} nS'
)
ax_raster.set_xlim(0, sim_dur)
ax_pop.fill_between(bin_centers, pop_rate / 1000, alpha=0.6, color='steelblue')
ax_pop.plot(bin_centers, pop_rate / 1000, color='steelblue', linewidth=0.8)
ax_pop.set_xlabel('Time (s)')
ax_pop.set_ylabel('Pop. rate (kHz)')
ax_pop.set_xlim(0, sim_dur)
plt.show()

---
## 4. The experimental recordings

We now load the "experimental" data. The data represents spike-sorted MEA recordings from a healthy- and a KCNQ2-deficient culture.
The data are in the same two-column format as the simulation output, so we
can use exactly the same code to visualise and analyse them.

Have a look at the raster plots (Notice the recordings are 3-minutes long, while the default simulation duration is 1 minute). What differences do you notice between
the two conditions? Also think again about the features. Are all informative for phenotyping these recordings?

In [ ]:
REC_DURATION = 180.0  # seconds, duration of the MEA recording

# Load spike trains -- same two-column format as simulation output
spike_data_healthy = np.loadtxt('Healtycontrol_MEArecording.csv', delimiter=',', skiprows=1)
spike_data_disease = np.loadtxt('KCNQ2patient_MEArecording.csv', delimiter=',', skiprows=1)

n_units_healthy = int(spike_data_healthy[:, 0].max()) + 1
n_units_disease = int(spike_data_disease[:, 0].max()) + 1

print(f'Healthy : {spike_data_healthy.shape[0]} spikes, {n_units_healthy} units')
print(f'Disease : {spike_data_disease.shape[0]} spikes, {n_units_disease} units')

In [ ]:
# Raster plot -- healthy
bin_edges_exp   = np.arange(0, REC_DURATION + bin_size, bin_size)
bin_centers_exp = (bin_edges_exp[:-1] + bin_edges_exp[1:]) / 2

def _pop_rate(spike_data, n_units, bin_edges):
    rm = np.zeros((n_units, builtins.len(bin_edges) - 1))
    for i in builtins.range(n_units):
        st = spike_data[spike_data[:, 0] == i, 1]
        if builtins.len(st) > 0:
            counts, _ = np.histogram(st, bins=bin_edges)
            rm[i] = counts / bin_size
    return np.sum(rm, axis=0)

pr_healthy = _pop_rate(spike_data_healthy, n_units_healthy, bin_edges_exp)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 4), sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
)
ax1.plot(spike_data_healthy[:, 1], spike_data_healthy[:, 0],
         '|k', markersize=2, alpha=0.5)
ax1.set_ylabel('Unit index')
ax1.set_title('Healthy culture')
ax1.set_xlim(0, REC_DURATION)
ax2.fill_between(bin_centers_exp, pr_healthy / 1000, alpha=0.6, color='steelblue')
ax2.plot(bin_centers_exp, pr_healthy / 1000, color='steelblue', linewidth=0.8)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Pop. rate (kHz)')
ax2.set_xlim(0, REC_DURATION)
plt.show()

In [ ]:
# Raster plot -- KCNQ2
pr_disease = _pop_rate(spike_data_disease, n_units_disease, bin_edges_exp)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 4), sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
)
ax1.plot(spike_data_disease[:, 1], spike_data_disease[:, 0],
         '|k', markersize=2, alpha=0.5)
ax1.set_ylabel('Unit index')
ax1.set_title('KCNQ2 culture')
ax1.set_xlim(0, REC_DURATION)
ax2.fill_between(bin_centers_exp, pr_disease / 1000, alpha=0.6, color='coral')
ax2.plot(bin_centers_exp, pr_disease / 1000, color='coral', linewidth=0.8)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Pop. rate (kHz)')
ax2.set_xlim(0, REC_DURATION)
plt.show()

In [ ]:
# Compute features for both experimental recordings
features_healthy = compute_features(spike_data_healthy, n_units_healthy, REC_DURATION)
features_disease = compute_features(spike_data_disease, n_units_disease, REC_DURATION)

print(f'  {"Feature":<25}  {"Healthy":>10}  {"Disease":>10}')
print(f'  {"-"*25}  {"-"*10}  {"-"*10}')
for name in features_healthy:
    print(f'  {name:<25}  {features_healthy[name]:>10.3f}  {features_disease[name]:>10.3f}')

---
## 5. Exercise: find the KCNQ2 parameters

The idea is of course to use computational models to find the biological difference between healthy and diseased recordings. Say here we measured all "fixed" parameters and we know they do not differ between our healthy and patient cultures. So we know the difference can only be in the unmeasured (or unmodelled) parameters: `B_ADAPT`, `POISSON_RATE`, and `G_SYN`. The parameters values provided above somewhat result in healthy-looking simulations, so the exercise now is to change these parameters to find the values such that your simulation reproduces the dynamics of the KCNQ2 recording.


**How to proceed:** go back to the parameters cell in Section 3, change
`B_ADAPT`, `POISSON_RATE`, and/or `G_SYN`, and re-run from there.
Use the comparison cell below to evaluate your result.

**Things to think about:**
- Which features are most useful for guiding your search to match simulation to experiment?
- Can you find a unique solution, or do multiple combinations give similar results?
- How would you approach this more systematically?
  What are the limitations of a manual or grid-based search? (you can try it out if you want)

In [ ]:
# Re-run this cell after each parameter change to compare your simulation
# against the KCNQ2 recording.

features_sim = compute_features(spike_data_sim, N_NEURONS, SIMULATION_TIME / second)

print(f'  {"Feature":<25}  {"KCNQ2 recording":>16}  {"Your simulation":>16}')
print(f'  {"-"*25}  {"-"*16}  {"-"*16}')
for name in features_disease:
    print(f'  {name:<25}  {features_disease[name]:>16.3f}  {features_sim[name]:>16.3f}')

---
## Optional assignments

**A. Visualise model internals**
Uncomment the `StateMonitor` lines in Section 4, re-run, and plot `v` and `w`
for a few neurons. Can you see the adaptation current building up during a
burst and terminating it? How does reducing `B_ADAPT` change this? You can also plot other variables like the synaptic current, and see how altering synaptic parameters alters the dynamics.

**B. Explore the fixed parameters**
Try changing `TAU_W`, `TAU_SYN`, `CONNECTION_PROB`, or any other parameters that interest you. How do these shape
the dynamics? Notice how many parameters this seemingly simple model already has.

**C. Add inhibitory neurons**
Most hiPSC cultures are predominantly excitatory, but some protocols produce
mixed networks. Extend the model with a small inhibitory population. How do the dynamics change? How many additional parameters are there?

**D. Define your own feature**
Think of something the current six features do not capture well. Implement it
and test whether it improves discrimination between healthy and disease.
You will be able to use it in notebook 2.